# NeuralGCM: control vs. K&C bogus for Typhoon Mawar (WP02 2023)

Runs Google's NeuralGCM (0.7° deterministic, `v1/deterministic_0_7_deg.pkl`)
twice from the same ERA5 analysis at **2023-05-26 06Z**, about a day before
Mawar's peak:

1. **control**: ERA5 regridded to the model grid, unaltered;
2. **bogus**: the same inputs with a Kwon & Cheong (2009) vortex matched to
   the JTWC best-track intensity.

Both runs go 72 h in 6 h steps. At each step the storm is located by
`tcinit.rollout_tracking.track_rollout`, which centres each box on the previous
step's position, and the two runs are compared against the best track.

**Run on:** a cee-compute GPU node in `jax-venv` (see `envs/README.md`). The
notebook reads ERA5 and the checkpoint from `gs://` anonymously, so the node
needs outbound network access (proxy set before Jupyter starts).

**Resolution.** 0.7° is NeuralGCM's finest checkpoint, but it is still a
~78 km Gaussian grid (TL255, 512 × 256) carried spectrally. Mawar's eye and
radius of maximum wind are far smaller than one grid cell, so both the
analysed storm and the bogus vortex are heavily smoothed: expect central
pressures tens of hPa above the best track, and treat this as a test of how
the model responds to the bogus, not of intensity skill.

### No surface fields: the "MSLP proxy"

NeuralGCM's inputs and outputs are pressure-level fields only (`u, v, z, t,
q` and cloud water on ERA5's 37 levels). It has no MSLP, 2 m temperature, or
10 m wind. tcinit's storm detection (`Snapshot.detect_storm`, used by
`snapshot_from_box`, `build_bogus` and `track_rollout`) needs `msl`, and the
K&C vortex needs a storm-mean 2 m temperature `T_0`. So `extract_box` below
adds proxy surface fields, under tcinit's canonical surface names, to every
box it returns:

| canonical | proxy | from |
|---|---|---|
| `msl` | $p_\mathrm{msl} \approx 1000\,\mathrm{hPa}\,\exp\!\big(\Phi_{1000} / (R_d\,T_{1000})\big)$ | hypsometric relation over the layer between 1000 hPa and sea level |
| `2t` | $T_{1000}$ | 1000 hPa temperature |
| `10u`, `10v` | $u_{1000}, v_{1000}$ | lowest pressure-level wind |

$\Phi_{1000}$ is the 1000 hPa geopotential (negative where the surface
pressure is below 1000 hPa, as in a typhoon core) and $R_d$ is
`tcinit.constants.R_D`. Over open ocean the layer is at most a few hundred
metres deep, so the isothermal approximation costs well under 1 hPa; cell 4b
checks the proxy against the real ERA5 MSLP regridded the same way. In model
output, NeuralGCM's decoder extrapolates `z` and `t` linearly below the model
surface and holds `u, v` constant from the lowest sigma level, so the proxy
stays defined in the storm core.

All pressures, winds, and tables below are therefore labelled **MSLP proxy**
and **1000 hPa wind**. The 1000 hPa wind has no surface-layer reduction, so
it runs higher than a true 10 m wind. The proxies live only in the tcinit
boxes. They are never written back into the model inputs (section 6 checks
this).

## 1. Configuration

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

# Pick one A100. Must be set before jax is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

PROJECT = Path("/projects/aurora")
BEST_TRACK_FILE = PROJECT / "best-tracks/westpac/bwp2023/bwp022023.dat"
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
MODEL_NAME = "v1/deterministic_0_7_deg.pkl"
MODEL_GCS = f"gs://neuralgcm/models/{MODEL_NAME}"
MODEL_CACHE = Path(f"/fgs/{os.environ['USER']}/causaltc/neuralgcm_models") / MODEL_NAME
OUT_DIR = Path(f"/fgs/{os.environ['USER']}/causaltc/demo_out/neuralgcm_mawar")

START_TIME = datetime(2023, 5, 26, 6)
STEP = timedelta(hours=6)
N_STEPS = 12  # 72 h
BOX_HALF_DEG = 10.0

# neuralgcm_backend.apply(match_surface_pressure=...): True (the default)
# writes the K&C surface low into geopotential so the encoder sees it; False
# gives warm core and winds only, for comparison (see section 6).
MATCH_SURFACE_PRESSURE = True

if not BEST_TRACK_FILE.exists():
    raise FileNotFoundError(BEST_TRACK_FILE)
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.parent.mkdir(parents=True, exist_ok=True)
print(f"output -> {OUT_DIR}")

In [ ]:
import pickle
from importlib import metadata

import gcsfs
import jax
import matplotlib.pyplot as plt
import neuralgcm
import numpy as np
import pandas as pd
import xarray as xr
# CHECK ON HPC: jax-venv has both `dinosaur` and `dinosaur-dycore` (same import
# name); the versions printed below say which one won. Both should provide
# these three modules.
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils

from tcinit import BestTrack
from tcinit.backends import neuralgcm_backend
from tcinit.constants import HPA_TO_PA, R_D
from tcinit.naming import NEURALGCM_VAR_MAP
from tcinit.plotting import plot_snapshot
from tcinit.rollout_snapshots import plot_rollout_comparison, plot_rollout_snapshots
from tcinit.rollout_tracking import build_bogus, snapshot_from_box, track_rollout

for dist in ("jax", "neuralgcm", "dinosaur", "dinosaur-dycore"):
    try:
        print(f"{dist:16s}", metadata.version(dist))
    except metadata.PackageNotFoundError:
        print(f"{dist:16s} (not installed)")
print("devices:", jax.devices())

## 2. Best track

In [ ]:
bt = BestTrack(BEST_TRACK_FILE)
bt_center = bt.get_location(START_TIME)
print(
    f"{bt.ID} at {START_TIME:%Y-%m-%d %HZ}: "
    f"({bt_center[0]:.1f}N, {bt_center[1]:.1f}E), "
    f"{bt.get_windspeed(START_TIME):.0f} m/s, "
    f"{bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa"
)

## 3. Load NeuralGCM

The model comes before the initial state here, unlike the Aurora notebook,
because its grid (`model.data_coords`) and variable lists define what to read
from ERA5 and how to regrid it. The checkpoint (about 1 GB) is cached on
`/fgs` after the first download. The checkpoint's auxiliary dataset also
carries the model's surface geopotential (orography · g), which the bogus
step in section 6 uses.

In [ ]:
if MODEL_CACHE.exists():
    with open(MODEL_CACHE, "rb") as f:
        ckpt = pickle.load(f)
else:
    gcs = gcsfs.GCSFileSystem(token="anon")
    with gcs.open(MODEL_GCS, "rb") as f:
        ckpt = pickle.load(f)
    with open(MODEL_CACHE, "wb") as f:
        pickle.dump(ckpt, f)

model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
# Model-grid surface geopotential (m2/s2), the field the encoder's orography
# module filters. Zero (sea level) if this checkpoint does not carry it.
aux_ds = xr.Dataset.from_dict(ckpt["aux_ds_dict"])
PHI_SFC = aux_ds["geopotential_at_surface"] if "geopotential_at_surface" in aux_ds else None
print("surface geopotential:", "from checkpoint" if PHI_SFC is not None else "zero (missing)")
del ckpt, aux_ds
print("inputs:  ", model.input_variables)
print("forcings:", model.forcing_variables)
print("grid:    ", model.data_coords.horizontal)
print("timestep:", model.timestep)

## 4. Initial state from ERA5

Following NeuralGCM's data-preparation guide: read one time from the ARCO
ERA5 Zarr store, conservatively regrid to the model's Gaussian grid
(`skipna=True` so SST and sea ice stay defined in coastal cells), then fill
the remaining land NaNs with nearest values. As in the NeuralGCM paper, the
forcings (SST, sea ice) are shifted back 24 h, and they are held fixed
(persistence) through the forecast.

ERA5's real MSLP and 10 m wind are regridded alongside, only to check the
proxy in 4b. They never enter the model.

In [ ]:
CHECK_VARS = [
    "mean_sea_level_pressure",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

full_era5 = xr.open_zarr(ERA5_ZARR, chunks=None, storage_options=dict(token="anon"))
era5 = (
    full_era5[model.input_variables + model.forcing_variables + CHECK_VARS]
    .pipe(
        xarray_utils.selective_temporal_shift,
        variables=model.forcing_variables,
        time_shift="24 hours",
    )
    .sel(time=[np.datetime64(START_TIME)])  # keep a length-1 time axis
    .compute()
)

era5_grid = spherical_harmonic.Grid(
    latitude_nodes=full_era5.sizes["latitude"],
    longitude_nodes=full_era5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(full_era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(full_era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True
)
model_vars = model.input_variables + model.forcing_variables
era5_model = xarray_utils.fill_nan_with_nearest(
    xarray_utils.regrid(era5[model_vars], regridder)
)
era5_check = xarray_utils.regrid(era5[CHECK_VARS], regridder).isel(time=0)

inputs = model.inputs_from_xarray(era5_model.isel(time=0))
input_forcings = model.forcings_from_xarray(era5_model.isel(time=0))
all_forcings = model.forcings_from_xarray(era5_model.head(time=1))  # persistence

LATS, LONS, LEVELS = neuralgcm_backend.axes_from_model(model, inputs)
print(f"grid {LONS.size} x {LATS.size}, {LEVELS.size} levels "
      f"({LEVELS.min():.0f}-{LEVELS.max():.0f} hPa)")
for k, v in inputs.items():
    print(f"  {k:36s} {np.shape(v)}")

### Boxes with proxy surface fields

`extract_box` cuts a box out of a NeuralGCM data dict (the regridded inputs,
or one decoded output step) with `neuralgcm_backend.extract_snapshot_dataset`.
The box keeps NeuralGCM's long-form names and native
`(level, longitude, latitude)` order, which tcinit's tracking functions accept
directly. To it are added the proxy `msl`, `2t`, `10u`, `10v` described at the
top, under their canonical names. `VAR_MAP` tells `snapshot_from_box`,
`build_bogus`, `track_rollout` and `plot_snapshot` where each canonical field
lives.

In [ ]:
P_LOW = float(LEVELS.max())  # 1000 hPa, the lowest pressure level
PROXIES = ("msl", "2t", "10u", "10v")
VAR_MAP = {**NEURALGCM_VAR_MAP, **{name: name for name in PROXIES}}


def add_surface_proxies(box):
    # MSLP proxy from the hypsometric relation between P_LOW and sea level.
    low = box.sel(level=P_LOW, drop=True)
    phi, t = low["geopotential"], low["temperature"]
    box["msl"] = P_LOW * HPA_TO_PA * np.exp(phi / (R_D * t))
    box["2t"] = t
    box["10u"] = low["u_component_of_wind"]
    box["10v"] = low["v_component_of_wind"]
    for name in PROXIES:
        box[name].attrs = {"note": f"proxy from {P_LOW:.0f} hPa fields"}
    return box


def extract_box(state, center):
    box = neuralgcm_backend.extract_snapshot_dataset(
        state,
        lats=LATS,
        lons=LONS,
        levels=LEVELS,
        box_center=center,
        box_size=BOX_HALF_DEG,
    )
    return add_surface_proxies(box)

### 4b. Sanity check: proxy vs. ERA5 MSLP, storm vs. best track

The proxy, computed from the regridded 1000 hPa fields, should agree with
ERA5's own MSLP regridded the same way to within about a hPa over the ocean,
and the detected storm should sit near the best-track position. The 0.25°
ERA5 minimum is printed to show how much the regridding alone fills the storm.

In [ ]:
box_ctrl0 = extract_box(inputs, bt_center)
snap_ctrl0 = snapshot_from_box(
    box_ctrl0, valid_time=START_TIME, search_center=bt_center, var_map=VAR_MAP
)

msl_true = (
    era5_check["mean_sea_level_pressure"]
    .sel(latitude=box_ctrl0.latitude, longitude=box_ctrl0.longitude, method="nearest")
    .transpose("latitude", "longitude")
    .values
)
diff = (box_ctrl0["msl"].transpose("latitude", "longitude").values - msl_true) / HPA_TO_PA
i, j = np.unravel_index(np.argmin(msl_true), msl_true.shape)
lat0, lon0 = bt_center
msl_025 = era5["mean_sea_level_pressure"].isel(time=0).sel(
    latitude=slice(lat0 + BOX_HALF_DEG, lat0 - BOX_HALF_DEG),  # ERA5 lat descends
    longitude=slice(lon0 - BOX_HALF_DEG, lon0 + BOX_HALF_DEG),
)
print(f"proxy - ERA5 MSLP in box: mean {diff.mean():+.2f}, "
      f"RMS {np.sqrt((diff ** 2).mean()):.2f}, max |{np.abs(diff).max():.2f}| hPa")
print(f"min MSLP: ERA5 0.25° {float(msl_025.min()) / HPA_TO_PA:.1f} hPa, "
      f"ERA5 regridded {msl_true.min() / HPA_TO_PA:.1f} hPa at "
      f"({box_ctrl0.latitude.values[i]:.2f}, {box_ctrl0.longitude.values[j]:.2f}), "
      f"proxy {snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa at "
      f"({snap_ctrl0.loc[0]:.2f}, {snap_ctrl0.loc[1]:.2f})")
print(f"best track: {bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa at "
      f"({lat0:.2f}, {lon0:.2f}); rdr {snap_ctrl0.rdr:.0f} km")

del era5, full_era5  # the 0.25° fields are no longer needed

## 5. Control initial state

In [ ]:
plot_snapshot(
    snap_ctrl0,
    box_ctrl0,
    var_map=VAR_MAP,
    title=f"Mawar, ERA5 on NeuralGCM 0.7° grid {START_TIME:%Y-%m-%d %HZ} "
    "(control initial state; MSLP and 10 m wind are 1000 hPa proxies)",
)
plt.show()

## 6. Bogus initial state

`build_bogus` detects the storm on the box (using the MSLP proxy), builds the
K&C vortex from the best-track intensity, and applies it through
`apply_atmos`. That function **drops the proxy surface fields**, then calls
`neuralgcm_backend.apply` (default `additive_z=False`). The returned box
therefore holds only `temperature`, `u/v_component_of_wind`,
`specific_humidity` and `geopotential`, which `neuralgcm_backend.write_back`
pastes into a copy of the input dict. (`write_back` also only iterates over
those five names, so a proxy could not leak through it anyway.) Cloud water,
`sim_time` and the forcings are untouched.

**Surface pressure.** NeuralGCM's encoder derives the model's surface
pressure from the input geopotential and orography
(`WeatherbenchToPrimitiveEncoder` calls `get_surface_pressure(z)`). The
decoder then rebuilds `z` from `t`, `q` and that surface pressure. So the
K&C surface low only reaches the model through `geopotential`. With its
default `match_surface_pressure=True`, `neuralgcm_backend.apply` does this in
two steps:

1. It updates each column's `z` hydrostatically for K&C's temperature change,
   so the warm core raises upper-level heights.
2. It shifts the column so the surface pressure, diagnosed the way the
   encoder does it (where `z` crosses the surface geopotential), equals the
   replacement blend $(1-b)\,p_\mathrm{ctrl} + b\,p_\mathrm{K\&C}$, with blend
   weight $b$.

Columns outside the blend are left bit-identical. The surface geopotential
for the box comes from the checkpoint (section 3); Mawar is over open ocean,
so it is about zero there anyway. Set `MATCH_SURFACE_PRESSURE = False` in
the config to compare against a bogus with warm core and winds but no
surface low.

In [ ]:
def apply_atmos(box, bv):
    # Drop the msl/2t/10u/10v proxies so they can never be written back.
    phi_sfc = None
    if PHI_SFC is not None:
        phi_sfc = (
            PHI_SFC.sel(latitude=box.latitude, longitude=box.longitude, method="nearest")
            .transpose("latitude", "longitude")
            .values
        )
    return neuralgcm_backend.apply(
        box.drop_vars(PROXIES),
        bv,
        match_surface_pressure=MATCH_SURFACE_PRESSURE,
        surface_geopotential=phi_sfc,
    )


_, bv, box_bogus0 = build_bogus(
    box_ctrl0,
    bt,
    START_TIME,
    search_center=bt_center,
    var_map=VAR_MAP,
    apply=apply_atmos,
)
assert not set(PROXIES) & set(box_bogus0.data_vars)
inputs_bogus = neuralgcm_backend.write_back(
    inputs, box_bogus0, lats=LATS, lons=LONS, levels=LEVELS
)

print(f"p_c  = {bv.p_c / HPA_TO_PA:.1f} hPa (best track)")
print(f"p_n  = {bv.p_n / HPA_TO_PA:.1f} hPa (env-mean MSLP proxy)")
print(f"K&C surface pressure min {bv.p_sfc_bogus_pa.min() / HPA_TO_PA:.1f} hPa")
print(f"T_0  = {bv.T_0:.1f} K (storm-mean 1000 hPa T)")
print(f"V_m  = {bv.V_m:.1f} m/s")
print(f"R_v  = {bv.R_v_km:.0f} km")

# Only the five atmospheric inputs may change, and only inside the box.
assert set(inputs_bogus) == set(inputs)
for k in inputs:
    a, b = np.asarray(inputs[k]), np.asarray(inputs_bogus[k])
    print(f"  {k:36s} max |change| {np.abs(b - a).max():.3g}")
    if k not in NEURALGCM_VAR_MAP.values():
        assert np.array_equal(a, b), k

box_check = extract_box(inputs_bogus, bt_center)
snap_bogus0 = snapshot_from_box(
    box_check, valid_time=START_TIME, search_center=bt_center, var_map=VAR_MAP
)
print(
    f"MSLP proxy min: control {snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa -> "
    f"bogus {snap_bogus0.mslp_min / HPA_TO_PA:.1f} hPa"
)
plot_snapshot(
    snap_bogus0,
    box_check,
    var_map=VAR_MAP,
    title=f"Mawar, ERA5 + K&C bogus on NeuralGCM 0.7° grid {START_TIME:%Y-%m-%d %HZ} "
    "(MSLP and 10 m wind are 1000 hPa proxies)",
)
plt.show()

## 7. Run and track both forecasts

Each run encodes its inputs, decodes the encoded state once for step 0, and
unrolls 12 × 6 h. Step 0 is the model's own reconstruction of the initial
condition, not the raw inputs: it shows what survives the encoder's
spectral truncation and learned correction (for the bogus run, how much of
the vortex the model actually received). `unroll` returns every step at
once, so the decoded fields are copied to the host and cut down to the five
variables tracking needs before `track_rollout` walks them.

In [ ]:
KEEP = list(NEURALGCM_VAR_MAP.values())  # the fields extract_box reads
TIMEDELTA = np.timedelta64(int(STEP.total_seconds()), "s")
rng_key = jax.random.key(42)  # ignored by the deterministic model


def run_forecast(model_inputs):
    # Returns [decoded initial state, step 1, ..., step N_STEPS] as numpy dicts.
    state = model.encode(model_inputs, input_forcings, rng_key)
    out0 = model.decode(state, input_forcings)
    _, traj = model.unroll(
        state,
        all_forcings,
        steps=N_STEPS,
        timedelta=TIMEDELTA,
        start_with_input=False,  # outputs at +6 h, ..., +72 h
    )
    out0 = {k: np.asarray(out0[k]) for k in KEEP}
    traj = {k: np.asarray(traj[k]) for k in KEEP}  # (time, level, lon, lat)
    return [out0] + [{k: traj[k][i] for k in KEEP} for i in range(N_STEPS)]


common = dict(
    cyclone_id=bt.ID,
    start_time=START_TIME,
    initial_center=bt_center,
    step=STEP,
    var_map=VAR_MAP,
)
ctrl = track_rollout(run_forecast(inputs), extract_box, **common)
bogus = track_rollout(
    run_forecast(inputs_bogus), extract_box, intervened=True, **common
)

ctrl.to_netcdf(OUT_DIR / "control")
bogus.to_netcdf(OUT_DIR / "bogus")
(OUT_DIR / "README_PROXY.txt").write_text(
    "NeuralGCM has no surface fields. In these snapshots _msl is an MSLP proxy\n"
    f"(hypsometric from {P_LOW:.0f} hPa z and t), _t2m is {P_LOW:.0f} hPa t, and\n"
    f"_u10/_v10 are {P_LOW:.0f} hPa winds. VMAX is the max {P_LOW:.0f} hPa wind.\n"
)

## 8. Compare with the best track

In [ ]:
cols = {
    "model_lead_hours": "lead_h",
    "model_MSLP": "MSLP_proxy",
    "model_VMAX": "VMAX_1000hPa",
    "position_error_km": "pos_err_km",
}
df_ctrl = ctrl.to_comparison_dataframe(bt)
df_bogus = bogus.to_comparison_dataframe(bt)
table = pd.concat(
    {
        "control": df_ctrl[list(cols)].rename(columns=cols),
        "bogus": df_bogus[list(cols)].rename(columns=cols).drop(columns="lead_h"),
        "best track": df_ctrl[["bt_MSLP", "bt_VMAX"]].rename(
            columns={"bt_MSLP": "MSLP", "bt_VMAX": "VMAX"}
        ),
    },
    axis=1,
)
table.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for df, label, style in ((df_ctrl, "control", "o-"), (df_bogus, "bogus", "^--")):
    lead = df["model_lead_hours"]
    axes[0].plot(lead, df["model_MSLP"], style, label=label)
    axes[1].plot(lead, df["model_VMAX"], style, label=label)
    axes[2].plot(lead, df["position_error_km"], style, label=label)
axes[0].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_MSLP"], "k-", lw=2, label="best track")
axes[1].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_VMAX"], "k-", lw=2, label="best track")
ylabels = (
    "min MSLP proxy (hPa)",
    "max 1000 hPa wind (kt)\n(best track: 10 m)",
    "position error (km)",
)
for ax, ylabel in zip(axes, ylabels):
    ax.set_xlabel("lead time (h)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(
    f"NeuralGCM 0.7°, Mawar from {START_TIME:%Y-%m-%d %HZ} "
    "(model MSLP and wind are 1000 hPa proxies)"
)
plt.tight_layout()
plt.show()

The PDFs below come from tcinit's generic plotting, so their panels say
"MSLP" and "10 m wind". For this model read those as the MSLP proxy and the
1000 hPa wind (`README_PROXY.txt` in the output directory says the same).

In [ ]:
plot_rollout_comparison(
    ctrl,
    bogus,
    OUT_DIR / "comparison_neuralgcm_mawar.pdf",
    best_track=bt,
    labels=("NeuralGCM control (MSLP proxy)", "NeuralGCM + K&C bogus (MSLP proxy)"),
)
plot_rollout_snapshots(ctrl, OUT_DIR / "rollout_neuralgcm_control.pdf", best_track=bt)
plot_rollout_snapshots(bogus, OUT_DIR / "rollout_neuralgcm_bogus.pdf", best_track=bt)
print(sorted(p.name for p in OUT_DIR.glob("*.pdf")))